In [ ]:
import os
#os.environ["HF_TOKEN"] = "your_own_token"


In [1]:
import logging
import os
from pathlib import Path
from typing import Optional, Dict
from dataclasses import dataclass

import torch
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.generation.logits_process import LogitsProcessor, LogitsProcessorList

# --- Constants ---
TASK_PROMPTS = {
    "normalization": "Instruct- Normalise the following text:\n",
    "transliteration": "Instruct- Transliterate the following text to Latin Script:\n",
    "punctuation": "Instruct- Punctuate the following text:\n",
}

# Mapping specific tasks to their required checkpoint files
CHECKPOINT_FILES = {
    "normalization": "weights/checkpoint_step_9900.pt",
    "default": "weights/checkpoint_step_32000.pt" # Used for punctuation and transliteration
}

# --- Configuration ---
@dataclass
class ModelConfig:
    """Configuration class."""
    model_name: str = "google/gemma-3-1b-pt"
    # Note: checkpoint_path is now handled dynamically per model loader
    token=os.environ.get("HF_TOKEN")
    trust_remote_code: bool = False
    bf16: bool = True
    attn_impl: str = "sdpa"

    # Generation Parameters
    max_new_tokens: int = 512
    num_beams: int = 1
    temperature: float = 0.0
    top_p: float = 0.9
    top_k: int = 0
    repetition_penalty: float = 1.0
    presence_penalty: float = 0.0
    frequency_penalty: float = 0
    no_repeat_ngram_size: int = 4
    seed: int = 42

# --- Custom Classes ---
class PenaltyLogitsProcessor(LogitsProcessor):
    def __init__(self, frequency_penalty: float = 0.0, presence_penalty: float = 0.0) -> None:
        if frequency_penalty < 0.0 or presence_penalty < 0.0:
            raise ValueError("Penalties must be non-negative")
        self.frequency_penalty = frequency_penalty
        self.presence_penalty = presence_penalty

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        if (self.frequency_penalty == 0.0 and self.presence_penalty == 0.0) or input_ids.numel() == 0:
            return scores

        for batch_idx in range(input_ids.size(0)):
            sequence = input_ids[batch_idx]
            if sequence.numel() == 0:
                continue

            unique_tokens, counts = torch.unique(sequence, return_counts=True)
            if self.frequency_penalty != 0.0:
                scores[batch_idx, unique_tokens] -= counts.to(scores.dtype) * self.frequency_penalty
            if self.presence_penalty != 0.0:
                scores[batch_idx, unique_tokens] -= self.presence_penalty

        return scores

# --- Helper Functions ---
def setup_logging() -> None:
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

def prepare_tokenizer(args: ModelConfig) -> AutoTokenizer:
    tokenizer = AutoTokenizer.from_pretrained(
        args.model_name,
        use_fast=True,
        token=os.environ.get("HF_TOKEN"),
        trust_remote_code=args.trust_remote_code,
    )
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def load_single_model(args: ModelConfig, checkpoint_path: str, model_id: str) -> AutoModelForCausalLM:
    """Loads a single instance of the model and robustly loads weights."""
    torch_dtype = torch.bfloat16 if args.bf16 else torch.float32

    logging.info(f"[{model_id}] Loading base model: {args.model_name}...")
    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        torch_dtype=torch_dtype,
        attn_implementation=args.attn_impl,
        trust_remote_code=args.trust_remote_code,
        token=os.environ.get("HF_TOKEN"), # Use the config token
        device_map="auto"
    )

    logging.info(f"[{model_id}] Overwriting weights from {checkpoint_path}...")

    # 1. Load the raw checkpoint
    checkpoint = torch.load(Path(checkpoint_path).expanduser().resolve(), map_location="cpu")

    # 2. Extract state_dict (handle if it's nested in "model" or "state_dict" key)
    if "model" in checkpoint:
        state_dict = checkpoint["model"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    # 3. Sanitize Keys (The Critical Fix)
    new_state_dict = {}
    for key, value in state_dict.items():
        new_key = key

        # Remove distributed training prefixes
        if new_key.startswith("module."):
            new_key = new_key[7:]

        # Remove torch.compile prefixes
        if new_key.startswith("_orig_mod."):
            new_key = new_key[10:]

        new_state_dict[new_key] = value

    # 4. Load state dict with verify
    missing, unexpected = model.load_state_dict(new_state_dict, strict=False)

    # 5. DIAGNOSTICS: Check if load actually worked
    if len(missing) > 0:
        # If we are missing > 50% of keys, the load definitely failed
        if len(missing) > len(new_state_dict) / 2:
            logging.error(f"[{model_id}] CRITICAL: Loaded weights failed! The keys likely do not match.")
            logging.error(f"[{model_id}] Checkpoint Key Example: {list(new_state_dict.keys())[0]}")
            logging.error(f"[{model_id}] Model Key Example: {list(model.state_dict().keys())[0]}")
            raise RuntimeError("Weight mismatch - Model is acting as base model.")
        else:
            logging.warning(f"[{model_id}] Missing keys (partial): {missing[:5]} ...")

    if unexpected:
        logging.warning(f"[{model_id}] Unexpected keys: {unexpected[:5]} ...")

    logging.info(f"[{model_id}] Weights loaded successfully.")
    model.eval()
    return model

def format_prompt(task: str, text: str) -> str:
    prefix = TASK_PROMPTS[task]
    return f"{prefix}{text.strip()}\n\nassistant: "

def generate(task: str, text: str, models_dict: Dict[str, AutoModelForCausalLM], tokenizer: AutoTokenizer, args: ModelConfig) -> str:
    # 1. Select the correct model based on the task
    if task == "normalization":
        model = models_dict["normalization"]
    else:
        # Default model handles transliteration and punctuation
        model = models_dict["default"]

    prompt = format_prompt(task, text)
    device = model.device

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}

    logits_processor = LogitsProcessorList()
    if args.frequency_penalty != 0.0 or args.presence_penalty != 0.0:
        logits_processor.append(
            PenaltyLogitsProcessor(
                frequency_penalty=args.frequency_penalty,
                presence_penalty=args.presence_penalty,
            )
        )

    generation_kwargs = {
        "max_new_tokens": args.max_new_tokens,
        "num_beams": max(1, args.num_beams),
        "do_sample": args.temperature > 0.0,
        "temperature": max(args.temperature, 1e-5) if args.temperature > 0.0 else None,
        "top_p": args.top_p,
        "top_k": args.top_k,
        "repetition_penalty": max(0.0, args.repetition_penalty),
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "no_repeat_ngram_size": max(0, args.no_repeat_ngram_size),
    }
    generation_kwargs = {k: v for k, v in generation_kwargs.items() if v is not None}

    with torch.no_grad():
        output_ids = model.generate(**inputs, logits_processor=logits_processor, **generation_kwargs)

    generated_sequence = output_ids[0]
    prompt_length = inputs["input_ids"].shape[-1]
    new_tokens = generated_sequence[prompt_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# --- Main Execution ---
setup_logging()

# 1. Configuration
config = ModelConfig(
    hf_token=os.environ.get("HF_TOKEN"), # Or hardcode if needed
    bf16=True,
    frequency_penalty=2.0
)

if config.hf_token:
    os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", config.hf_token)

# 2. Download Checkpoints
logging.info("Downloading checkpoints from Hugging Face Hub...")
paths = {}

try:
    # Download Normalization checkpoint (9900)
    paths["normalization"] = hf_hub_download(
        repo_id="SSneha2005/NPT",
        filename=CHECKPOINT_FILES["normalization"],
        repo_type="model",
        token=os.environ.get("HF_TOKEN")
    )
    # Download Default checkpoint (32000)
    paths["default"] = hf_hub_download(
        repo_id="SSneha2005/NPT",
        filename=CHECKPOINT_FILES["default"],
        repo_type="model",
        token=os.environ.get("HF_TOKEN")

    )
    logging.info("Checkpoints downloaded successfully.")
except Exception as e:
    logging.error(f"Failed to download checkpoints: {e}")
    raise e

# 3. Load Models into VRAM
tokenizer = prepare_tokenizer(config)
MODELS_CACHE = {}

# Load Normalization Model
MODELS_CACHE["normalization"] = load_single_model(config, paths["normalization"], "Norm_Model_9900")

# Load Default Model
MODELS_CACHE["default"] = load_single_model(config, paths["default"], "Default_Model_32000")

logging.info(f"System Ready. Model loaded: {list(MODELS_CACHE.keys())}")



2026-01-11 06:05:49,344 [INFO] Downloading checkpoints from Hugging Face Hub...


weights/checkpoint_step_9900.pt:   0%|          | 0.00/6.00G [00:00<?, ?B/s]

weights/checkpoint_step_32000.pt:   0%|          | 0.00/6.00G [00:00<?, ?B/s]

2026-01-11 06:10:40,775 [INFO] Checkpoints downloaded successfully.


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

2026-01-11 06:11:21,252 [INFO] We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

2026-01-11 06:11:22,982 [INFO] [Norm_Model_9900] Overwriting weights from /root/.cache/huggingface/hub/models--SSneha2005--NPT/snapshots/71a3dd2f774e1628a7a88bc8288b1b59bbae0bcc/weights/checkpoint_step_9900.pt...
2026-01-11 06:11:47,614 [INFO] [Norm_Model_9900] Weights loaded successfully.
2026-01-11 06:11:47,944 [INFO] [Default_Model_32000] Loading base model: google/gemma-3-1b-pt...
2026-01-11 06:11:48,317 [INFO] We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
2026-01-11 06:11:56,864 [INFO] [Default_Model_32000] Overwriting weights from /root/.cache/huggingface/hub/models--SSneha2005--NPT/snapshots/71a3dd2f774e1628a7a88bc8288b1b59bbae0bcc/weights/checkpoint_step_32000.pt...
2026-01-11 06:12:21,677 [INFO] [Default_Model_32000] Weights loaded successfully.
2026-01-11 06:12:21,992 [INFO] System Ready. Model loaded: ['normalization', 'default']


In [2]:
# --- Inputs ---
input_batch = {
    "normalization": [
        "Report suggests 3.14 million packages delayed, a rise of 2 lakh.",
        "अंतिम तिथि 24-05-2026, दोपहर 2:30 आई॰ऍस॰टी॰ से पहले है।"
    ],
    "punctuation": [
        "इसे भी पढ़ें इस बॉलर ने एक ओवर में लिया 6 विकेट सभी को किया बोल्ड",
        "With the Sankranthi releases in full swing there are no major releases which will hit the screens this week so the lack of releases is all set to help the current films at the ticket window"
    ],
    "transliteration": [
        "अब, यहाँ एक बार फिर से, यह ग्रिपिंग पैड (gripping pads) या ग्रिपिंग जॉ (gripping jaws) है जिसकी मदद से मैं इस विशेष वस्तु को पकडने जा रहा हूँ, माना की वस्तु यहाँ है।",
        "అఖిల భారత జొన్న సంస్కరణ ప్రణాళికలో రెండు సంవత్స్రరాల వరకు జొన్న నాటే ప్రముఖ ప్రదేశాలలో ఉన్న కేంద్రాలలో పరీక్షలు చేయబడ్డాయి.",
        "उन्होंने कहा कि 1960 के दशक के बाद ज़्यादातर लोगों ने ऐसी नीतियां बनाई जो लोगों को पसंद आये पर मोदी जी ने ऐसी नीतियाँ बनाने का काम किया जो लोगों के लिए अच्छी हों और बिना डरे यह काम किया।"
    ]
}

# --- Execution & Printing ---
try:
    print("STARTING BATCH PROCESS...\n")

    for task_name, texts in input_batch.items():

        # 1. Validate Task
        if task_name not in TASK_PROMPTS:
            print(f"!!! Error: Task '{task_name}' not found. Skipping. !!!\n")
            continue

        # 2. Print Task Header
        header = f" TASK: {task_name.upper()} "
        print(f"{'='*20}{header}{'='*20}")

        # Ensure 'texts' is a list
        if isinstance(texts, str):
            texts = [texts]

        # 3. Process and Print Each Text
        for i, text in enumerate(texts, 1):
            # Pass task_name, raw text, and the MODELS_CACHE dictionary
            # Note: format_prompt is handled inside generate now
            output = generate(task_name, text, MODELS_CACHE, tokenizer, config)

            # Print Result with clear separation
            print(f"Item #{i}")
            print(f"Input:  {text}")
            print(f"Output: {output}")
            print("-" * 60) # Separator between items

        print("\n") # Spacing between tasks

    print("=== BATCH COMPLETE ===")

except NameError as e:
    print(f"Error: {e}. Make sure MODELS_CACHE and generate() are defined.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


STARTING BATCH PROCESS...

==================== TASK: NORMALIZATION ====================
Item #1
Input:  Report suggests 3.14 million packages delayed, a rise of 2 lakh.
Output: Report suggests three point one four million packages delayed a rise of two lakh.
------------------------------------------------------------
Item #2
Input:  अंतिम तिथि 24-05-2026, दोपहर 2:30 आई॰ऍस॰टी॰ से पहले है।
Output: अंतिम तारीख चौबीस मई दो हजार छब्बीस, दोपहर दो बजकर तीस मिनट आई॰ऍएस॰टी॰ के पहले है।
------------------------------------------------------------


==================== TASK: PUNCTUATION ====================
Item #1
Input:  इसे भी पढ़ें इस बॉलर ने एक ओवर में लिया 6 विकेट सभी को किया बोल्ड
Output: इसे भी पढ़ें, इस बॉलर ਨੇ एक ओवर में لیا 6 विकेट (सभी को) किया बोल्ड।
------------------------------------------------------------
Item #2
Input:  With the Sankranthi releases in full swing there are no major releases which will hit the screens this week so the lack of releases is all set to help the curr

In [ ]:
import sys
import torch

# 1. Redefine generate() with your exact snippet
def generate(task, text, models_dict, tokenizer, args):
    # Select the model based on task
    if task == "normalization":
        model = models_dict["normalization"]
    else:
        model = models_dict["default"]

    # Create prompt
    prompt = format_prompt(task, text)

    # ────────── YOUR EXACT SNIPPET START ──────────
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(model.device)

    # 3. Generate
    with torch.no_grad():
        generated = model.generate(
            input_ids      = enc["input_ids"],
            attention_mask = enc["attention_mask"],
            max_new_tokens = 64,
            pad_token_id   = tokenizer.pad_token_id,
            eos_token_id   = tokenizer.eos_token_id
        )

    # 4. Decode and Slice (Remove the input prompt from the output)
    # Get the length of the input tokens to slice the output
    input_len = enc["input_ids"][0].shape[0]
    new_tokens = generated[0][input_len:]

    output_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # ────────── YOUR SNIPPET END ──────────

    return output_text

# 2. Run Interactive Loop
# Validate dependencies exist
required_vars = ["MODELS_CACHE", "tokenizer", "config", "TASK_PROMPTS"]
if not all(var in globals() for var in required_vars):
    print("❌ Error: Previous cell not loaded. Please run the model loading cell first.")
else:
    print("✅ Logic updated to your custom snippet.")
    print(f"👉 Available Tasks: {list(TASK_PROMPTS.keys())}")

    current_task = "transliteration"
    print(f"\n🔥 Active Task: {current_task.upper()}")
    print("💡 Commands: Type 'task' to switch modes, or 'exit' to quit.\n")

    while True:
        try:
            text = input(f"\n[{current_task}] Input: ")

            if text.lower() in ['exit', 'quit', 'q']:
                print("Exiting...")
                break

            if text.lower() == 'task':
                print(f"Options: {list(TASK_PROMPTS.keys())}")
                new_task = input("Enter new task name: ").strip().lower()
                if new_task in TASK_PROMPTS:
                    current_task = new_task
                    print(f"✅ Switched to: {current_task.upper()}")
                else:
                    print(f"❌ Invalid task. Options are: {list(TASK_PROMPTS.keys())}")
                continue

            if not text.strip():
                continue

            # Run with the new generate function
            output = generate(
                task=current_task,
                text=text,
                models_dict=MODELS_CACHE,
                tokenizer=tokenizer,
                args=config
            )

            print(f"🔸 Output: {output}")
            print("-" * 50)

        except KeyboardInterrupt:
            print("\nStopped by user.")
            break
        except Exception as e:
            print(f"❌ Error during generation: {e}")

✅ Logic updated to your custom snippet.
👉 Available Tasks: ['normalization', 'transliteration', 'punctuation']

🔥 Active Task: TRANSLITERATION
💡 Commands: Type 'task' to switch modes, or 'exit' to quit.


[transliteration] Input: ఎం చేస్తున్నావ్
🔸 Output: em chestunnaav
--------------------------------------------------

[transliteration] Input: 'नमस्ते'
🔸 Output: 'Namaste'
--------------------------------------------------

Stopped by user.
